## 🔧 Environment Setup

In [ ]:
!nvidia-smi

Fri Mar 20 22:06:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Installing ultralytics

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.8 MB/s eta 0:00:00


## 🦏 Training the YOLOv8 Animal Detection Model

We are training a YOLOv8 small model (`yolov8s`) on our African wildlife dataset, which contains four animal classes: **rhino, zebra, elephant, and buffalo**.  

The training command below will:

1. Load the dataset configuration from `african-wildlife.yaml` (contains paths to images and class names).  
2. Start training from the pre-trained `yolov8s.pt` weights.  
3. Run for **30 epochs**, updating the model weights each time it sees all the images.  
4. Resize all images to **640×640 pixels** for consistent input size.  
5. Process **8 images per batch** to balance GPU memory and training speed.  

The model will learn to detect these animals in images and videos, and the best weights will be saved for inference and tracking.

In [ ]:
!yolo detect train data=african-wildlife.yaml model=yolov8s.pt epochs=30 imgsz=640 batch=8

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=african-wildlife.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False,

## 📂 Mount Google Drive

- Mount Google Drive to save/load models, images, videos, and outputs.
- Enables persistent storage beyond Colab's temporary environment.

In [ ]:
from ultralytics import YOLO
from google.colab import drive

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
#mount google drive
drive.mount('/content/drive')

Mounted at /content/drive


## 🐾 Load Trained Model

In [ ]:
#load trained animal detection model
model = YOLO("/content/drive/MyDrive/best.pt")

## 🖼️ Run Object Detection on an Image

In [ ]:
#image detection
image_path = "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/images/test/1 (171).jpg"
results = model(image_path, save=True)  # saves output in /content/runs/detect/predict


image 1/1 /content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/images/test/1 (171).jpg: 512x640 1 buffalo, 1140.3ms
Speed: 16.9ms preprocess, 1140.3ms inference, 48.9ms postprocess per image at shape (1, 3, 512, 640)
Results saved to /content/runs/detect/predict


## 🎥 Track Animals in Videos

- Tracks multiple animals across frames in a video and assigns consistent IDs.
- Saves tracked videos in `/content/runs/track/predict`.
- Handles multiple videos with a loop for batch processing.


In [ ]:
#video tracking
video_list = [
    ("rhino_and_baby", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Rhino and Baby.mp4"),
    ("elephants", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Elephants.mp4"),
    ("zabras", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Zebras.mp4")
]

#save demo videos
for name, path in video_list:
    model.track(path, save=True)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 342ms
Prepared 1 package in 22ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/418) /content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/vide

## 📡 Stream Tracking Results in Memory

- Streams tracking results frame-by-frame for analysis or visualization.
- Allows inspecting boxes, IDs, and classes without saving the video.
- Useful for building datasets for further analysis.

In [ ]:
results = model.track(track_video_path, stream=True)

for r in results:
    print(r.boxes)

Streaming output truncated to the last 5000 lines.
xyxyn: tensor([[0.2682, 0.5159, 0.4237, 0.6751],
        [0.6609, 0.3279, 0.9998, 0.6505]])
video 1/1 (frame 156/418) /content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/Rhino and Baby.mp4: 384x640 2 rhinos, 10.7ms
ultralytics.engine.results.Boxes object with attributes:

cls: tensor([2., 2.])
conf: tensor([0.9413, 0.9421])
data: tensor([[5.1359e+02, 5.5649e+02, 8.1494e+02, 7.3097e+02, 1.0000e+00, 9.4133e-01, 2.0000e+00],
        [1.2717e+03, 3.5556e+02, 1.9199e+03, 7.0308e+02, 2.0000e+00, 9.4210e-01, 2.0000e+00]])
id: tensor([1., 2.])
is_track: True
orig_shape: (1080, 1920)
shape: torch.Size([2, 7])
xywh: tensor([[ 664.2656,  643.7294,  301.3508,  174.4862],
        [1595.8401,  529.3220,  648.1825,  347.5185]])
xywhn: tensor([[0.3460, 0.5960, 0.1570, 0.1616],
        [0.8312, 0.4901, 0.3376, 0.3218]])
xyxy: tensor([[ 513.5902,  556.4863,  814.9410,  730.9725],
        [1271.7489,  355.5628, 1919.9314,  703

## 📊 Build Tracking Dataset from Videos

- Extracts per-frame information for each tracked animal:  
  - `frame number`, `animal ID`, `class name`, `confidence`, `bounding box coordinates`.
- Computes additional features: center coordinates, movement, behavior, and welfare alerts.
- Generates a DataFrame for analysis and Power BI visualization.

In [ ]:
import pandas as pd
import numpy as np

# list of videos (same as saved demos)
video_list = [
    ("rhino_and_baby", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Rhino and Baby.mp4"),
    ("elephants", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Elephants.mp4"),
    ("zabras", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Zebras.mp4")
]

rows = []

# track and build dataset
for video_name, track_video_path in video_list:
    print(f"Processing video: {video_name}")

    # track video in memory
    results = model.track(track_video_path, stream=True)

    for frame_idx, r in enumerate(results):
        if r.boxes is None or r.boxes.id is None:
            continue

        for i in range(len(r.boxes)):
            x1, y1, x2, y2 = r.boxes.xyxy[i].tolist()
            conf = float(r.boxes.conf[i])
            cls = int(r.boxes.cls[i])
            track_id = int(r.boxes.id[i])
            class_name = model.names[cls]

            rows.append([
                video_name,
                frame_idx,
                track_id,
                class_name,
                conf,
                x1, y1, x2, y2
            ])

# create dataframe
df = pd.DataFrame(rows, columns=[
    "video", "frame", "animal_id", "class_name", "confidence", "x1", "y1", "x2", "y2"
])

Processing video: rhino_and_baby
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 293ms
Prepared 1 package in 85ms
Installed 1 package in 4ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


video 1/1 (frame 1/418) /content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Rhino and Baby.mp4: 384x640 2 rhinos, 888.2ms
video 1/1 (frame 2/418) /content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Rhino and Baby.mp4: 384x640 2 rhinos, 475.0ms
video 1/1 (frame 3/418) /content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Rhino and Baby.mp4: 384x640 2 rhinos, 508.6ms
video 1/1 (frame 4/418) /content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Rhino and Baby.mp4: 384x640 2 rhino

## 🚶 Calculate Animal Movement and Classify Behavior

- Computes the center of bounding boxes.  
- Calculates frame-to-frame movement (Euclidean distance).  
- Classifies animal behavior based on movement: `resting`, `walking`, `running`.  
- Adds a welfare alert if movement is low.

In [ ]:
# Calculate center of bounding box
df["center_x"] = (df["x1"] + df["x2"]) / 2
df["center_y"] = (df["y1"] + df["y2"]) / 2

In [ ]:
# Sort data
df = df.sort_values(["animal_id","frame"])

# Previous positions
df["prev_x"] = df.groupby("animal_id")["center_x"].shift(1)
df["prev_y"] = df.groupby("animal_id")["center_y"].shift(1)

# Movement calculation
df["movement"] = np.sqrt(
    (df["center_x"] - df["prev_x"])**2 +
    (df["center_y"] - df["prev_y"])**2
)

df["movement"] = df["movement"].fillna(0)

In [ ]:
def classify_behaviour(m):

    if m < 2:
        return "resting"
    elif m < 20:
        return "walking"
    else:
        return "running"

df["behaviour"] = df["movement"].apply(classify_behaviour)

In [ ]:
def welfare_alert(row):

    if row["movement"] < 10:
        return "Low Activity"

    return "Normal"

df["welfare_alert"] = df.apply(welfare_alert, axis=1)

## 💾 Save Tracking Data for Analysis

- Exports processed tracking data to a CSV file.  
- CSV can be imported into Power BI, Excel, or SQL for reporting and dashboards.


In [ ]:
df.to_csv("/content/drive/MyDrive/Colab Notebooks/Animal Detection/animal_tracking.csv", index=False)